In [3]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [1]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    if message.value['amount'] > 3000:
        print(f"to jest duza transakcja {message.value} UWAGA")
    
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź amount > 1000, jeśli tak — wypisz ALERT

Writing consumer_filter.py


In [2]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-enrich-group',  # inny group_id
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    event = message.value

    amount = event.get('amount', 0)

    if amount > 3000:
        event['risk_level'] = "HIGH"
    elif amount > 1000:
        event['risk_level'] = "MEDIUM"
    else:
        event['risk_level'] = "LOW"

    print(f"{event}")

Writing consumer_enrich.py


In [1]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    event = message.value

    store = event.get('store', 'UNKNOWN')
    amount = event.get('amount', 0)

    store_counts[store] += 1

    total_amount[store] = total_amount.get(store, 0) + amount

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n--- PODSUMOWANIE ---")
        print(f"{'Sklep':<15} {'Liczba':<10} {'Suma':<10} {'Średnia':<10}")

        for store in store_counts:
            count = store_counts[store]
            total = total_amount[store]
            avg = total / count if count > 0 else 0

            print(f"{store:<15} {count:<10} {total:<10.2f} {avg:<10.2f}")

Writing consumer_count.py


In [2]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    'count': 0,
    'total': 0,
    'min': float('inf'),
    'max': float('-inf')
})

msg_count = 0

for message in consumer:
    event = message.value

    category = event.get('category', 'UNKNOWN')
    amount = event.get('amount', 0)

    stats[category]['count'] += 1
    stats[category]['total'] += amount
    stats[category]['min'] = min(stats[category]['min'], amount)
    stats[category]['max'] = max(stats[category]['max'], amount)

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n--- STATYSTYKI PER KATEGORIA ---")
        print(f"{'Kategoria':<15} {'Liczba':<10} {'Suma':<12} {'Min':<10} {'Max':<10}")

        for category, data in stats.items():
            print(
                f"{category:<15} "
                f"{data['count']:<10} "
                f"{data['total']:<12.2f} "
                f"{data['min']:<10.2f} "
                f"{data['max']:<10.2f}"
            )

Writing consumer_stats.py


In [ ]:
## zad dom

In [3]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
from collections import defaultdict, deque
import json
import time

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-anomaly-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_events = defaultdict(deque)

window=60
threshold = 3

for message in consumer:
    event=message.value
    user_id = event.get('user_id','unknown')
    raw_ts = event.get('timestamp')

    try:
        timestamp = float(raw_ts)
    except (TypeError, ValueError):
        try:
            timestamp = datetime.fromisoformat(raw_ts).timestamp()
        except Exception:
            timestamp = time.time()

    user_events[user_id].append(timestamp)
    while user_events[user_id] and timestamp - user_events[user_id][0] > window:
        user_events[user_id].popleft()

    if len(user_events[user_id]) > threshold:
        print(f"ALERT: user {user_id} wykonał >3 transakcje w 60s")
        print(f"Zdarzenia: {list(user_events[user_id])}")

Writing consumer_anomaly.py
